In [1]:
import pandas as pd
import matplotlib
import numpy
import re

df = pd.read_csv(
    r'/Users/urielle.zach/Data-Preprocessing/globalmart_dirty_orders_2000.csv', 
    keep_default_na=False,
    na_values=['-', '#N/A', 'N/A', 'NULL', '?', 'unknown', 'NONE', 'na', 'None', 'none', 'NaN', 'NA', 'nan', ''], # catch all null values before parsing
)   

df.index = df.index + 2 ## shifted index to align with csv

columns = [
    'record_id', 'source_system', 'customer_id', 'customer_name', 
    'customer_email', 'phone_raw', 'age_raw', 'gender_raw', 
    'signup_date_raw', 'order_id', 'order_date_raw', 'ship_date_raw', 
    'delivery_date_raw', 'customer_timezone', 'country_raw', 'city_raw', 
    'postal_code_raw', 'product_category_raw', 'product_name_raw', 
    'sku_raw', 'quantity_raw', 'unit_price_raw', 'currency_raw', 
    'discount_raw', 'shipping_cost_raw', 'item_weight_raw', 
    'payment_method_raw', 'order_status_raw', 'returned_flag_raw', 
    'return_reason_raw', 'satisfaction_score_raw', 'loyalty_points_raw', 
    'site_visits_last30_raw', 'support_tickets_90d_raw', 
    'distance_to_warehouse_km_raw', 'campaign_source_raw', 
    'customer_segment_raw', 'support_ticket_date_raw', 
    'complaint_text_raw', 'ingestion_batch', 'data_source_note'
]

*** TASK 1 ***

In [ ]:
def returnRows(df):
    return f"Total row count: {len(df)}"

def returnDatatype(df):
    return df.dtypes

def checkMissingValues(df, columns):
    null_rows = df[columns].isnull().sum() ## returns how many null rows are there in the series
    missing_values = null_rows[null_rows > 0] ## filters all boolean values that are greater than 0 to be passed through {missing_values} variable
    total_rows = len(df)
    
    percentages = (missing_values / total_rows) * 100
    print("--------------------Missing Rows--------------------")
    summary_df = pd.DataFrame({
        'Missing Count': missing_values,
        'Percentage (%)': percentages.round(2)  
    })

    return summary_df

## specifies which columns and row has null values
def specifyWhere(df, columns):
    missing_locations = {}
    for col in columns:
        null_indeces = df[df[col].isnull()].index.tolist()

        if null_indeces:
            missing_locations[col] = null_indeces

    return missing_locations

## if {value} of {column_name} HAS duplicates:
    ## do: 
        ## return the row where the {value} is duplicated
        ## filter all {value} to show all true
        ## show the number of duplicated {value} in a column

def checkSpecificColumnDuplicates(df, column_name):
    duplicate_mask = df.duplicated(subset=[column_name], keep=False) ## checks every value in index to check if {value} has duplicates
    duplicated_rows = df[duplicate_mask] ## passes all duplicates into duplicated_rows
    duplicated_rows = duplicated_rows.dropna(subset=[column_name]) ## drops all values containing {NaN}
    duplicates_clean_view = duplicated_rows[['record_id', column_name]]

    print(f"Total amount of duplicated rows: {len(duplicates_clean_view)}")
    return duplicates_clean_view

def genderRemap(df):
    print("unique:")
    print(f"{df["gender_raw"].unique()}\n")
    clean_source = df["gender_raw"].str.upper().str.strip()

    mapping = {'NB': 'Non-Binary', 
               'NON-BINARY': 'Non-Binary',
               'M': 'Male', 
               'F':'Female', 
               'MALE': 'Male',
               'FEMALE': 'Female',
               'PREFER NOT TO SAY': 'Prefer not to say'
               }
    
    print(f"dict: {mapping} \n")
    df["gender_raw"] = clean_source.replace(mapping)
    
    ## recheck
    print("remapped gender_raw")
    return df["gender_raw"]

def sourceRemap(df):
    print("unique: ")
    print(f"{df["source_system"].unique()}\n")
    clean_source = df["source_system"].str.upper().str.strip()

    mapping = {
        'STORE': 'In-Store', 
        'IN-STORE': 'In-Store',
        'MOBILE': 'Mobile App', 
        'CALL CENTRE': 'Call Center',
        'CALL CENTER': 'Call Center',
        'PARTNER-API': 'Partner-API',
        'PARTNER API': 'Partner-API', 
        'MARKETPLACE': 'Marketplace',
        'WEB': 'Web',
        'MOBILE-APP': 'Mobile App',
        'MOBILE APP': 'Mobile App',
        'APP': 'Mobile App',
        'MARKET PLACE': 'Marketplace'
    }
    
    print(f"dict: {mapping} \n")
    df["source_system"] = clean_source.replace(mapping)

    print("remapped source_system")
    return df["source_system"]


## TODO: 
## INVALID DETECTION (PRIORITY!!!!)
## COUNTRY_REMAP (hopefully without regex)
## CITY_RAW
## PRODUCT_CATEGORY_RAW
## CAMPAIGN_SOURCE_RAW
## CUSTOMER_SEGMENT_RAW

def countryRemap(df):
    print("unique: ")
    print(df["source_system"].unique())
    
    clean_source = df["source_system"].str.upper().str.strip()
    mapping = {
        'STORE': 'IN-STORE', 
        'MOBILE': 'MOBILE APP', 
        'CALL CENTRE': 'CALL CENTER',
        'PARTNER API': 'PARTNER-API', 
        'MARKETPLACE': 'MARKET PLACE' 
    }
    
    print(f"dict: {mapping} \n")
    df["source_system"] = clean_source.replace(mapping)

    print("remapped source_system")
    return df["source_system"]

## email validation regex: ^[\w-\.]+@([\w-]+\.)+[\w-]{2,4}$     

In [7]:
## all commands functions run here are pre-normalization
sourceRemap(df)
df["source_system"].unique()

unique: 
<StringArray>
[ 'Mobile App',         'Web',    'IN-STORE', 'Marketplace', 'Call Center',
         'APP', 'PARTNER-API',           nan]
Length: 8, dtype: str

dict: {'STORE': 'In-Store', 'IN-STORE': 'In-Store', 'MOBILE': 'Mobile App', 'CALL CENTRE': 'Call Center', 'CALL CENTER': 'Call Center', 'PARTNER API': 'Partner-API', 'MARKETPLACE': 'Marketplace', 'WEB': 'Web', 'MOBILE-APP': 'Mobile App', 'MOBILE APP': 'Mobile App', 'MARKET PLACE': 'Marketplace'} 

remapped source_system


<StringArray>
[ 'Mobile App',         'Web',    'In-Store', 'Marketplace', 'Call Center',
         'APP', 'PARTNER-API',           nan]
Length: 8, dtype: str